In [31]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict,Literal
from dotenv import load_dotenv


In [32]:
load_dotenv();


In [33]:
model=ChatOpenAI(model='gpt-4o-mini')

In [34]:
class quadraticState(TypedDict):
    a:int
    b:int
    c:int
    equation:str
    discriminant:float
    result:str

In [44]:
def show_eq(state:quadraticState)->quadraticState:
    eq=f"{state['a']}x2{state['b']}x{state['c']}"
    return {'equation':eq}

def cal_dis(state:quadraticState)->quadraticState:
    dis=state['b']**2 - (4*(state['a']*state['c']))

    return {'discriminant':dis}

def real_root(state:quadraticState)->quadraticState:
    root1=(-state["b"] + state["discriminant"]**0.5)/(2*state['a'])
    root2=(-state["b"] - state["discriminant"]**0.5)/(2*state['a'])

    result=f"The roots are {root1} and {root2}"

    return {'result':result}


def repeated_root(state:quadraticState)->quadraticState:
    root=(-state["b"])/(2*state['a'])
   
    result=f"Only repeated root is {root}"

    return {'result':result}


def no_real_root(state:quadraticState)->quadraticState:
    result=f"No real roots"

    return {'result':result}


def conditional(state:quadraticState)->Literal['real_root','no_real_root','repeated_root']:
    if state['discriminant'] > 0:
        return 'real_root'
    elif state['discriminant'] < 0:
        return 'no_real_root'
    else:
        return 'repeated_root'

In [45]:
graph =StateGraph(quadraticState)

graph.add_node('show_eq',show_eq)
graph.add_node('cal_dis',cal_dis)
graph.add_node('no_real_root',no_real_root)
graph.add_node('real_root',real_root)
graph.add_node('repeated_root',repeated_root)


graph.add_edge(START,'show_eq')
graph.add_edge('show_eq','cal_dis')
graph.add_conditional_edges('cal_dis',conditional)
graph.add_edge('no_real_root',END)
graph.add_edge('real_root',END)
graph.add_edge('repeated_root',END)


workflow=graph.compile()

In [49]:
inital_state={'a':6,'b':5,'c':2}
final_state=workflow.invoke(inital_state)
print(final_state)


{'a': 6, 'b': 5, 'c': 2, 'equation': '6x25x2', 'discriminant': -23, 'result': 'No real roots'}
